In [ ]:
!pip install datasets -q
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

**LOAD AND EXPLORE THE DATASET**

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("jason23322/high-accuracy-email-classifier")
df_new = dataset['train'].to_pandas()

print("Shape:", df_new.shape)
print("\nColumns:", df_new.columns.tolist())
print("\nCategory Distribution:")
print(df_new['category'].value_counts())
print("\nSample:")
print(df_new.head(3))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df_old = pd.read_pickle("/content/drive/MyDrive/email_data.pkl")
print("Enron Shape:", df_old.shape)
print("Enron Columns:", df_old.columns.tolist())

**COMBINE TWO DATASETS (HUGGING FACE + ENRON)**

In [ ]:
label_map = {
    'spam': 'Spam',
    'promotions': 'Promotions',
    'forum': 'Personal',
    'social_media': 'Personal',
    'updates': 'Support',
    'verify_code': 'Support'
}

df_new['label'] = df_new['category'].map(label_map)
df_new_clean = df_new[['text', 'label']].rename(columns={'text': 'processed_text'})

df_old_clean = df_old[['processed_text', 'label']]

df_combined = pd.concat([df_new_clean, df_old_clean], ignore_index=True)
df_combined = df_combined.dropna().reset_index(drop=True)

df_combined.to_pickle("/content/drive/MyDrive/email_combined.pkl")

print("Combined Shape:", df_combined.shape)
print("\nLabel Distribution:")
print(df_combined['label'].value_counts())
print("\nSample:")
print(df_combined.head(3))

**DATA PREPROCESSING**

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    words = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return " ".join(words)

In [ ]:
df_combined['processed_text'] = df_combined['processed_text'].apply(preprocess)

df_combined.to_pickle("/content/drive/MyDrive/email_combined.pkl")

print("Preprocessing done!")
print(df_combined[['processed_text', 'label']].head(3))

**TRAIN/TEST SPLIT + CHECK DISTRIBUTION**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_combined['label_enc'] = le.fit_transform(df_combined['label'])

In [ ]:
print("Label Mapping:")
for i, c in enumerate(le.classes_):
    print(f"  {i} → {c}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_combined['processed_text'],
    df_combined['label_enc'],
    test_size=0.2,
    random_state=42,
    stratify=df_combined['label_enc']
)

print("\nTrain size:", len(X_train))
print("Test size:", len(X_test))
print("\nTrain Distribution:")
print(df_combined.loc[X_train.index, 'label'].value_counts())

**FINE-TUNE DISTILLBERT**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.metrics import classification_report
import torch
import numpy as np

In [ ]:
# Dataset class
class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=128)
        self.labels = list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

In [ ]:
# Load DistilBERT
ft_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
ft_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4)


In [ ]:
# Prepare datasets
train_dataset = EmailDataset(X_train.tolist(), y_train.tolist(), ft_tokenizer)
test_dataset = EmailDataset(X_test.tolist(), y_test.tolist(), ft_tokenizer)


In [ ]:
# Train
args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=ft_model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

In [ ]:
# Evaluate
preds = trainer.predict(test_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(y_test.tolist(), y_pred, target_names=le.classes_))

**SAVE THE MODEL**

In [ ]:
import os, pickle

save_path = "/content/drive/MyDrive/email_classifier_v2/"
os.makedirs(save_path, exist_ok=True)

ft_model.save_pretrained(save_path + "distilbert_finetuned")
ft_tokenizer.save_pretrained(save_path + "distilbert_finetuned")
pickle.dump(le, open(save_path + "label_encoder.pkl", "wb"))

print("New model saved successfully!")
print(os.listdir(save_path))